In [ ]:
# @title ⚙ Setup — run first {display-mode: "form"}
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'ipywidgets', 'plotly', '--quiet'], capture_output=True)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/workshop_data/clips'
import warnings; warnings.filterwarnings('ignore')
print('Drive mounted.'  )


In [ ]:
# @title 📦 Load data {display-mode: "form"}
import json, re, os, urllib.request
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, Image, clear_output
from pathlib import Path

with open(f'{DRIVE_ROOT}/workshop_manifest.json') as _f:
    MANIFEST = json.load(_f)

CALLS    = MANIFEST['calls']
ENDPOINT = MANIFEST['google_apps_script_endpoint']
QUARTERS = [c['quarter'] for c in CALLS]

def dpath(fname): return f'{DRIVE_ROOT}/{fname}'

RETURNS    = pd.read_csv(dpath(MANIFEST['files']['returns']))
GAAP       = pd.read_csv(dpath(MANIFEST['files']['gaap_metrics']))
KPI        = pd.read_csv(dpath(MANIFEST['files']['nonfin_kpis']))
GRAND_MEAN = pd.read_csv(dpath(MANIFEST['files']['grand_mean_final'])).iloc[0]
SUMMARY    = pd.read_csv(dpath(MANIFEST['files']['all_calls_summary']))

FEATURES = {}
IMG_MFST = {}
for _c in CALLS:
    _q = _c['quarter']
    FEATURES[_q] = pd.read_csv(dpath(_c['files']['windowed_features']))
    IMG_MFST[_q] = pd.read_csv(dpath(_c['files']['image_manifest']))

CLR = {
    'lm_positive':  '#22c55e',
    'lm_negative':  '#ef4444',
    'lm_neutral':   '#9ca3af',
    'fou_above':    '#f59e0b',
    'fou_below':    '#3b82f6',
    'ret_positive': '#22c55e',
    'ret_negative': '#ef4444',
    'analyst_text': '#fbbf24',
    'musk_text':    '#60a5fa',
}

STATE = {
    'team_name': '', 'registered': False,
    'r1_submitted': False, 'r2_submitted': False, 'r3_submitted': False,
    'r2_chip_shift_used': False, 'r3_chip_shift_used': False,
    'chips': {q: 3 for q in QUARTERS},
}

# Anonymous labels (non-chronological — prevents chart cross-referencing)
ANON_MAP = {
    "2024Q1": "Call A",
    "2023Q1": "Call B",
    "2022Q4": "Call C",
    "2024Q2": "Call D",
}
ANON_QUARTERS = [ANON_MAP[q] for q in QUARTERS]
print('Data loaded.')


In [ ]:
# @title 🔧 Utilities {display-mode: "form"}
display(HTML("<style>\n  .ag-card{background:#1e293b;border:1px solid #334155;border-radius:8px;padding:16px;margin:8px;font-family:'Georgia',serif;color:#e2e8f0;}\n  .ag-card h4{color:#818cf8;margin:0 0 8px 0;font-size:1.1em;}\n  .ag-excerpt-box{background:#0f172a;border-left:3px solid #334155;padding:10px 14px;border-radius:4px;font-size:0.88em;line-height:1.6;margin:8px 0;}\n  .ag-analyst{color:#fbbf24;}\n  .ag-musk{color:#60a5fa;}\n  .ag-label{color:#94a3b8;font-size:0.8em;text-transform:uppercase;letter-spacing:0.05em;margin-bottom:2px;}\n  .ag-gauge-pos{color:#22c55e;font-size:1.4em;font-weight:bold;}\n  .ag-gauge-neg{color:#ef4444;font-size:1.4em;font-weight:bold;}\n  .ag-gauge-neu{color:#9ca3af;font-size:1.4em;font-weight:bold;}\n  .ag-section-hdr{background:linear-gradient(90deg,#1e293b,#0f172a);border-left:4px solid #818cf8;padding:10px 16px;margin:20px 0 10px 0;font-family:'Georgia',serif;color:#e2e8f0;font-size:1.1em;}\n  .ag-status-box{background:#1e293b;border:1px solid #334155;border-radius:6px;padding:12px;font-family:monospace;font-size:0.85em;color:#94a3b8;}\n  .ag-table{border-collapse:collapse;width:100%;}\n  .ag-table th{background:#334155;color:#e2e8f0;padding:6px 10px;text-align:left;font-size:0.85em;}\n  .ag-table td{padding:5px 10px;font-size:0.85em;border-bottom:1px solid #1e293b;}\n  .ag-table tr:nth-child(even){background:#1e293b;}\n  .ag-table tr:nth-child(odd){background:#0f172a;}\n  .ag-arrow-up{color:#22c55e;font-weight:bold;}\n  .ag-arrow-down{color:#ef4444;font-weight:bold;}\n  .ag-arrow-flat{color:#9ca3af;}\n  .ag-na{color:#475569;font-style:italic;}\n  .ag-word-counter{font-size:0.78em;color:#94a3b8;text-align:right;margin-top:2px;}\n</style>\n"))

def post_to_script(payload):
    if not ENDPOINT or 'ENDPOINT_URL_HERE' in ENDPOINT:
        print('Apps Script endpoint not configured (offline mode).')
        return {'success': True, 'message': 'offline'}
    import requests as _req, json as _j
    r = _req.post(ENDPOINT, json=payload, timeout=20,
                  headers={'Content-Type': 'application/json'})
    return _j.loads(r.text)

def get_status(round_number):
    try: return post_to_script({'action':'get_status','round':round_number})
    except: return {'submitted_teams':[],'total':'?'}

def render_sentiment_bars(ax, sentences, title=''):
    nets  = [s['net'] for s in sentences]
    cats  = [s['category'] for s in sentences]
    colors = [CLR['lm_positive'] if c=='positive' else
              CLR['lm_negative'] if c=='negative' else CLR['lm_neutral']
              for c in cats]
    ax.bar(range(len(sentences)), nets, color=colors, width=0.7)
    ax.axhline(0, color='#475569', linewidth=0.8)
    ax.set_xticks(range(len(sentences)))
    ax.set_xticklabels([f'S{i+1}' for i in range(len(sentences))],
                       fontsize=7, color='#94a3b8')
    ax.set_ylabel('LM net', fontsize=7, color='#94a3b8')
    ax.tick_params(axis='y', labelsize=7, colors='#94a3b8')
    ax.set_facecolor('#0f172a')
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    for sp in ['bottom','left']: ax.spines[sp].set_color('#334155')
    if title: ax.set_title(title, fontsize=8, color='#818cf8', pad=4)

FOU_MEAN = float(GRAND_MEAN['fraction_of_unvoiced_mean'])

RADAR_FEATS  = ['mean_pitch','mean_intensity','number_of_periods',
                'fraction_of_unvoiced','number_of_voice_breaks',
                'jitter_local','shimmer_local','mean_autocorrelation','mean_nhr']
RADAR_LABELS = ['Pitch','Intensity','Voiced rate','Silence','Hesitation',
                'Freq stability','Amp stability','Vocal clarity','Breathiness']

def z_score(val, feat):
    mu  = GRAND_MEAN.get(f'{feat}_mean', np.nan)
    sig = GRAND_MEAN.get(f'{feat}_std',  np.nan)
    if not sig or np.isnan(sig): return 0.0
    return (val - mu) / sig

def make_silence_map(c):
    q = c['quarter']
    df = FEATURES[q].copy()
    kq = c['key_question']['window_index']
    kq_text = (c['key_question']['analyst_text_preview'] or '')[:80]
    colors = [CLR['fou_above'] if v > FOU_MEAN else CLR['fou_below']
              for v in df['fraction_of_unvoiced']]
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df['window_index'], y=df['fraction_of_unvoiced'],
        marker_color=colors,
        hovertemplate='<b>Window %{x}</b><br>FoU: %{y:.3f}<extra></extra>'))
    fig.add_hline(y=FOU_MEAN, line_dash='dot', line_color='#94a3b8',
        annotation_text=f'Mean ({FOU_MEAN:.3f})',
        annotation_font_color='#94a3b8', annotation_font_size=10)
    if kq is not None and kq < len(df):
        fig.add_vline(x=kq, line_dash='dash', line_color='#818cf8',
            annotation_text='Key Q', annotation_font_color='#818cf8',
            annotation_font_size=10)
    fig.update_layout(
        title=dict(text=f'{q} — Silence patterns across the Q&A',
                   font=dict(color='#e2e8f0', size=13)),
        xaxis=dict(title='5-min window', color='#94a3b8', gridcolor='#1e293b'),
        yaxis=dict(title='Fraction of unvoiced (0–1)', range=[0,1],
                   color='#94a3b8', gridcolor='#1e293b'),
        plot_bgcolor='#0f172a', paper_bgcolor='#1e293b',
        font=dict(color='#e2e8f0'), height=260,
        margin=dict(l=50,r=20,t=50,b=40), showlegend=False)
    return fig

def make_radar(c):
    q = c['quarter']
    row = SUMMARY[SUMMARY['call'] == f'TSLA_{q}']
    if len(row) == 0: return go.Figure()
    row = row.iloc[0]
    MAX_Z = 3.0
    def norm(z): return max(0, min(1, (z + MAX_Z) / (2 * MAX_Z)))
    r_call = []
    flags  = []
    for feat in RADAR_FEATS:
        val = row.get(f'{feat}_mean_musk', np.nan)
        if pd.isna(val): val = 0.0
        z = z_score(val, feat)
        r_call.append(norm(z))
        flags.append(abs(z) > 1.0)
    r_grand = [0.5] * len(RADAR_FEATS)
    labels  = RADAR_LABELS
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=r_call+[r_call[0]], theta=labels+[labels[0]],
        fill='toself', fillcolor='rgba(96,165,250,0.15)',
        line=dict(color='#60a5fa', width=2), name=q))
    fig.add_trace(go.Scatterpolar(r=r_grand+[r_grand[0]], theta=labels+[labels[0]],
        fill='none', line=dict(color='#475569', width=1, dash='dot'),
        name='Cross-call average'))
    for i,(feat,flag) in enumerate(zip(RADAR_FEATS, flags)):
        if flag:
            fig.add_trace(go.Scatterpolar(r=[r_call[i]], theta=[labels[i]],
                mode='markers', marker=dict(color='#f59e0b', size=10),
                showlegend=False))
    fig.update_layout(
        polar=dict(bgcolor='#0f172a',
            radialaxis=dict(visible=True, range=[0,1], color='#475569',
                            gridcolor='#334155', tickfont_size=7),
            angularaxis=dict(color='#94a3b8', gridcolor='#334155')),
        showlegend=True,
        legend=dict(font=dict(color='#94a3b8', size=9), bgcolor='rgba(0,0,0,0)'),
        paper_bgcolor='#1e293b',
        title=dict(text=f'{q} — Feature radar (Musk responses)',
                   font=dict(color='#e2e8f0', size=12)),
        height=320, margin=dict(l=40,r=40,t=50,b=20))
    return fig

def arrow_cell(val, yoy, unavail=False):
    if unavail or (isinstance(val, float) and np.isnan(val)):
        return '<span class="ag-na">n/a</span>'
    yoy_str = ''
    if yoy is not None and not (isinstance(yoy, float) and np.isnan(yoy)):
        if yoy > 2:
            yoy_str = f' <span class="ag-arrow-up">↑{yoy:.1f}%</span>'
        elif yoy < -2:
            yoy_str = f' <span class="ag-arrow-down">↓{abs(yoy):.1f}%</span>'
        else:
            yoy_str = ' <span class="ag-arrow-flat">→</span>'
    return f'{val:,.0f}{yoy_str}'

def _c(row, col, yoy_col):
    v = row.get(col)
    unavail = bool(row.get(f'{col}_unavailable', pd.isna(v) if v is not None else True))
    if pd.isna(v): return '<span class="ag-na">n/a</span>'
    yoy = row.get(yoy_col)
    return arrow_cell(v, yoy if (yoy is not None and not pd.isna(yoy)) else None, unavail)

def build_gaap_table(fy):
    rows = GAAP[GAAP['fy'] == fy]
    if len(rows) == 0: return '<p>No GAAP data for this year.</p>'
    r = rows.iloc[0].to_dict()
    gm = r.get('gross_margin_pct','')
    gm_str = f'{gm:.1f}%' if isinstance(gm, float) and not np.isnan(gm) else 'n/a'
    body = (
        f'<tr><td>Total Revenue (USD m)</td><td>{_c(r,"total_revenue","total_revenue_yoy_pct")}</td></tr>'
        f'<tr><td>Gross Profit (USD m)</td><td>{_c(r,"gross_profit","gross_margin_pct_yoy_pct")}</td></tr>'
        f'<tr><td>Gross Margin %</td><td>{gm_str}</td></tr>'
        f'<tr><td>Operating Income (USD m)</td><td>{_c(r,"operating_income","operating_income_yoy_pct")}</td></tr>'
        f'<tr><td>Net Income (USD m)</td><td>{_c(r,"net_income","net_income_yoy_pct")}</td></tr>'
        f'<tr><td>EPS Diluted</td><td>{_c(r,"eps_diluted","eps_diluted_yoy_pct")}</td></tr>'
    )
    return (f'<table class="ag-table"><tr><th>GAAP Metric — FY{fy}</th>'
            f'<th>Value (YoY)</th></tr>{body}</table>')

def build_kpi_table(fy):
    rows = KPI[KPI['fy'] == fy]
    if len(rows) == 0: return '<p>No KPI data.</p>'
    r = rows.iloc[0].to_dict()
    sc = '<span class="ag-na">n/a — not reported in 10-K</span>'
    body = (
        f'<tr><td>Vehicle Deliveries</td><td>{_c(r,"vehicle_deliveries","vehicle_deliveries_yoy_pct")}</td></tr>'
        f'<tr><td>Vehicle Production</td><td>{_c(r,"vehicle_production","vehicle_production_yoy_pct")}</td></tr>'
        f'<tr><td>Energy Storage (GWh)</td><td>{_c(r,"energy_storage_gwh","energy_storage_gwh_yoy_pct")}</td></tr>'
        f'<tr><td>Solar Deployed (MW)</td><td>{_c(r,"solar_deployed_mw","solar_deployed_mw_yoy_pct")}</td></tr>'
        f'<tr><td>Supercharger Stations</td><td>{sc}</td></tr>'
        f'<tr><td>Supercharger Connectors</td><td>{sc}</td></tr>'
    )
    return (f'<table class="ag-table"><tr><th>Non-financial KPI — FY{fy}</th>'
            f'<th>Value (YoY)</th></tr>{body}</table>')

print('Utilities ready.')


<div style='background:#0f172a;padding:28px 32px;border-radius:10px;border:1px solid #334155;font-family:Georgia,serif;color:#e2e8f0;'>
<h1 style='color:#818cf8;margin:0 0 6px 0;font-size:1.8em;'>The Analyst&#8217;s Edge</h1>
<h2 style='color:#94a3b8;margin:0 0 20px 0;font-weight:normal;font-size:1.1em;'>Multimodal Analysis of Tesla Earnings Calls &nbsp;&middot;&nbsp; AG952 Week 10</h2>
<p style='line-height:1.8;margin-bottom:14px;'>It is the morning after a Tesla earnings call. You are a buy-side analyst at a mid-size asset manager. Your portfolio manager is waiting for your note before the market opens. You have three sources of evidence: the text of the Q&amp;A session, a set of infographics from the most recent annual report, and acoustic measurements of Elon Musk&#8217;s voice during the call. Your task is to integrate these signals and make a prediction about how Tesla&#8217;s stock will perform over the next five trading days.</p>
<p style='line-height:1.8;margin-bottom:14px;'>You will work through three rounds, each introducing one new modality. In each round you will update your prediction and, crucially, explain your reasoning. At the end, the actual outcomes are revealed and the team whose predictions and reasoning best withstand the evidence wins the session.</p>
<p style='line-height:1.8;color:#94a3b8;font-size:0.92em;margin:0 0 16px 0;'><strong style='color:#e2e8f0;'>Four earnings calls</strong> have been selected from the Tesla transcript archive (2022&#8211;2024), labelled <strong style='color:#818cf8;'>Call A&#8211;D</strong> in a randomised order. You will not know which real-world quarter each label corresponds to until the reveal at the end &#8212; this prevents you from using the stock chart to look up the answer.</p><div style='background:#1e293b;border:1px solid #334155;border-radius:8px;padding:16px 20px;margin-top:4px;'><p style='color:#818cf8;font-weight:bold;margin:0 0 10px 0;font-size:1em;'>&#127922; How chips work</p><p style='line-height:1.8;margin:0 0 10px 0;'>Allocate <strong>1, 2, or 3 chips</strong> per call before submitting Round 1 &#8212; there is no fixed total. Think of chips as conviction: the more chips you place on a call, the more you win if you are right &#8212; and the more you lose if you are wrong.</p><ul style='line-height:2;margin:0 0 10px 0;padding-left:20px;'><li>Allocate <strong>1, 2, or 3 chips</strong> per call independently. You do not need to reach any particular total.</li><li>A correct prediction earns you <strong>+chips</strong> points. A wrong prediction costs you <strong>&#8722;chips</strong> points.</li><li>After Round 2 and again after Round 3 you may <strong>shift 1 chip</strong> from one call to another (once per round, optional, irreversible) if the new evidence changes your conviction.</li></ul><p style='line-height:1.8;margin:0;color:#94a3b8;font-size:0.9em;'><strong style='color:#e2e8f0;'>Example:</strong> you put 3 chips on Call A and predict &#8220;Stock rose&#8221;. If the stock did rise, you score +3. If it fell, you score &#8722;3. A team that concentrates chips on calls it reads correctly will outscore a team that spreads chips evenly but guesses randomly.</p></div>
</div>


In [ ]:
display(HTML('<div class="ag-section-hdr">Step 1 of 4 &nbsp;&middot;&nbsp; Register your team</div>'))
team_input = widgets.Text(placeholder='e.g. Team Hawkins',
    description='Team name:', layout=widgets.Layout(width='340px'),
    style={'description_width': '90px'})
reg_btn    = widgets.Button(description='Register', button_style='primary',
    layout=widgets.Layout(width='120px'))
reg_status = widgets.HTML()

def on_register(b):
    name = team_input.value.strip()
    if not name:
        reg_status.value = '<span style="color:#ef4444;">Please enter a team name.</span>'
        return
    STATE['team_name'] = name
    try:
        resp = post_to_script({'action':'register_team','team_name':name})
        msg = 'already registered' if resp.get('message')=='already_registered' else 'registered'
        if resp.get('success'):
            STATE['registered'] = True
            reg_status.value = (f'<span style="color:#22c55e;">✓ <strong>{name}</strong>'
                                f' {msg}. You are ready to begin.</span>')
            team_input.disabled = True; reg_btn.disabled = True
        else:
            reg_status.value = f'<span style="color:#ef4444;">Registration failed.</span>'
    except Exception:
        STATE['registered'] = True
        reg_status.value = (f'<span style="color:#f59e0b;">⚠ Offline mode — '
                            f'team name saved locally as <strong>{name}</strong>.</span>')

reg_btn.on_click(on_register)
display(widgets.HBox([team_input, reg_btn]))
display(reg_status)


In [ ]:
display(HTML('<div class="ag-section-hdr">Tesla stock price 2021&#8211;2024</div>'))
display(HTML("<div style='background:#1e293b;border:1px solid #334155;border-radius:8px;padding:16px 22px;margin:12px 0;font-family:Georgia,serif;color:#e2e8f0;line-height:1.8;'><p style='margin:0 0 10px 0;'>The chart below shows Tesla's daily closing share price from January 2021 to December 2024. Each dashed vertical line marks the trading day of a Tesla earnings call. The lines are colour-coded by outcome: <span style='color:#22c55e;font-weight:bold;'>green</span> indicates that Tesla's stock <em>outperformed</em> the S&amp;P 500 over the five trading days following that call (positive abnormal return); <span style='color:#ef4444;font-weight:bold;'>red</span> indicates that it <em>underperformed</em> the S&amp;P 500 over the same window (negative abnormal return).</p><p style='margin:0 0 10px 0;'><strong>Abnormal return</strong> is defined here as the cumulative TSLA return over days +1 to +5 after the call, minus the cumulative S&amp;P 500 return over the same window. It isolates the market&#8217;s reaction to the call itself from broader market moves on those days. A positive abnormal return means the call was net good news for Tesla relative to the market; a negative abnormal return means it was net bad news.</p><p style='margin:0 0 6px 0;'>Four of these calls have been selected for today&#8217;s workshop. For each one, you will receive three sources of evidence in sequence &#8212; the text of the Q&amp;A, financial infographics, and acoustic measurements of Elon Musk&#8217;s voice &#8212; and must predict whether that call&#8217;s line would be <span style='color:#22c55e;font-weight:bold;'>green</span> or <span style='color:#ef4444;font-weight:bold;'>red</span> on this chart. The outcomes are locked and hidden until every team has submitted all three rounds.</p></div>\n"))
try:
    display(Image(filename=f'{DRIVE_ROOT}/TSLA_price_timeline_workshop.png', width=900))
except Exception:
    display(HTML('<p style="color:#94a3b8;">Timeline image not found.</p>'))
display(HTML("<div style='background:#1e293b;border:1px solid #334155;border-radius:8px;padding:14px 20px;margin:12px 0;font-family:Georgia,serif;'>\n<p style='color:#818cf8;margin:0 0 10px 0;font-weight:bold;'>Colour grammar &#8212; consistent throughout this notebook</p>\n<table style='border-collapse:collapse;'>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#22c55e;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Green</span></td><td style='color:#e2e8f0;font-size:0.88em;padding:4px 0;'>LM positive sentiment &nbsp;|&nbsp; Positive stock return</td></tr>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#ef4444;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Red</span></td><td style='color:#e2e8f0;font-size:0.88em;'>LM negative sentiment &nbsp;|&nbsp; Negative stock return</td></tr>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#9ca3af;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Grey</span></td><td style='color:#e2e8f0;font-size:0.88em;'>LM neutral sentiment</td></tr>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#f59e0b;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Amber</span></td><td style='color:#e2e8f0;font-size:0.88em;'>Above-average fraction of unvoiced (more silence / hesitation)</td></tr>\n<tr><td style='padding:4px 16px 4px 0;'><span style='background:#3b82f6;padding:3px 10px;border-radius:4px;color:#fff;font-size:0.85em;'>&#9632; Blue</span></td><td style='color:#e2e8f0;font-size:0.88em;'>Below-average fraction of unvoiced (more voiced / confident)</td></tr>\n</table>\n<p style='color:#94a3b8;font-size:0.82em;margin:10px 0 0 0;'><strong style='color:#e2e8f0;'>Chip allocation:</strong> Allocate 1, 2, or 3 chips per call before submitting Round 1 — no fixed total required. Higher allocation amplifies your score for that call. You may shift one chip once in Round 2 and once in Round 3.</p>\n</div>\n"))


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:18px 24px;border-radius:8px;border-left:4px solid #818cf8;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<h2 style='color:#818cf8;margin:0 0 4px 0;'>Round 1 &#8212; The Text</h2>
<p style='color:#94a3b8;margin:0;font-size:0.95em;'>Read the Q&amp;A excerpts below. Use only the text to make your first predictions. Each excerpt shows the highest-divergence question-answer pair from the call transcript.</p>
</div>


In [ ]:
# ── helpers ──────────────────────────────────────────────────────────────────
def _fmt_sentences(sentences, base_col):
    """Render sentences with subtle LM sentiment tinting."""
    parts = []
    for s in sentences:
        txt = s["text"].strip()
        if not txt:
            continue
        cat = s.get("category", "neutral")
        if cat == "positive":
            bg, border = "rgba(34,197,94,0.15)", "#22c55e"
        elif cat == "negative":
            bg, border = "rgba(239,68,68,0.15)", "#ef4444"
        else:
            bg, border = "transparent", "transparent"
        parts.append(
            f'<span style="background:{bg};border-bottom:1px solid {border};' +
            f'border-radius:2px;padding:0 2px;">{txt}</span> '
        )
    return "".join(parts)

def make_qa_html(c):
    exc   = c["excerpt"]
    q     = c["quarter"]
    anon  = ANON_MAP.get(q, q)
    a_html = _fmt_sentences(exc["analyst_sentences"], "#fbbf24")
    m_html = _fmt_sentences(exc["musk_sentences"],    "#60a5fa")
    net    = exc["musk_net_sent"]
    if net > 0.005:
        gcol, gsym, glbl = "#22c55e", f"+{net:.3f}", "net positive"
    elif net < -0.005:
        gcol, gsym, glbl = "#ef4444", f"{net:.3f}",  "net negative"
    else:
        gcol, gsym, glbl = "#9ca3af", f"{net:.3f}",  "near-neutral"
    return (
        f'<div style="background:#1e293b;border:1px solid #334155;border-radius:8px;' +
        f'padding:16px;font-family:Georgia,serif;color:#e2e8f0;">' +
        f'<h4 style="color:#818cf8;margin:0 0 4px 0;">{anon}</h4>' +
        f'<div style="color:#94a3b8;font-size:0.75em;text-transform:uppercase;' +
        f'letter-spacing:0.05em;margin-bottom:8px;">Analyst: {exc["analyst_speaker"]}</div>' +
        f'<div style="background:#0f172a;border-left:3px solid #fbbf24;border-radius:4px;' +
        f'padding:10px 14px;margin-bottom:10px;max-height:160px;overflow-y:auto;">' +
        f'<div style="color:#fbbf24;font-size:0.72em;text-transform:uppercase;' +
        f'letter-spacing:0.07em;margin-bottom:6px;">Analyst question</div>' +
        f'<div style="font-size:0.88em;line-height:1.8;color:#e2e8f0;">{a_html}</div></div>' +
        f'<div style="background:#0f172a;border-left:3px solid #60a5fa;border-radius:4px;' +
        f'padding:10px 14px;margin-bottom:10px;max-height:280px;overflow-y:auto;">' +
        f'<div style="color:#60a5fa;font-size:0.72em;text-transform:uppercase;' +
        f'letter-spacing:0.07em;margin-bottom:6px;">Elon Musk response</div>' +
        f'<div style="font-size:0.88em;line-height:1.8;color:#e2e8f0;">{m_html}</div></div>' +
        f'<div style="font-size:0.82em;margin-top:4px;">' +
        f'<span style="color:#94a3b8;">Aggregate LM sentiment (Musk):&nbsp;</span>' +
        f'<span style="color:{gcol};font-weight:bold;">{gsym}</span>' +
        f'&nbsp;<span style="color:#94a3b8;">({glbl})</span>' +
        f'<span style="color:#475569;font-size:0.85em;">&nbsp;&nbsp;Sentence tints: ' +
        f'<span style="background:rgba(34,197,94,0.25);padding:1px 5px;border-radius:2px;">positive</span> ' +
        f'<span style="background:rgba(239,68,68,0.25);padding:1px 5px;border-radius:2px;">negative</span> ' +
        f'no tint = neutral</span></div></div>'
    )

def make_sent_chart(c):
    out = widgets.Output()
    with out:
        fig, axes = plt.subplots(1, 2, figsize=(7, 2.0))
        fig.patch.set_facecolor('#1e293b')
        render_sentiment_bars(axes[0], c["excerpt"]["analyst_sentences"], "Analyst sentences")
        render_sentiment_bars(axes[1], c["excerpt"]["musk_sentences"],    "Musk sentences")
        plt.tight_layout(pad=0.4); plt.show(); plt.close()
    return out

# ── per-call input widgets ────────────────────────────────────────────────────
chip_sliders   = {}
pred_dropdowns = {}
reason_areas   = {}
word_counters  = {}

for _c in CALLS:
    _q = _c["quarter"]
    chip_sliders[_q] = widgets.IntSlider(
        value=3, min=1, max=3, step=1,
        description="Chips:", style={"description_width": "55px"},
        layout=widgets.Layout(width="260px"))
    pred_dropdowns[_q] = widgets.Dropdown(
        options=["-- select --", "Stock rose", "Stock fell"],
        description="Predict:", style={"description_width": "55px"},
        layout=widgets.Layout(width="230px"))
    reason_areas[_q] = widgets.Textarea(
        placeholder="Your rationale (max 30 words) …",
        layout=widgets.Layout(width="100%", height="90px"))
    word_counters[_q] = widgets.HTML(
        value='<div class="ag-word-counter">0 / 30 words</div>')
    def _mk_wc(_q_=_q):
        def _upd(ch):
            wc  = len(ch["new"].split())
            col = "#ef4444" if wc > 30 else "#94a3b8"
            word_counters[_q_].value = (
                f'<div class="ag-word-counter" style="color:{col};">{wc} / 30 words</div>')
        return _upd
    reason_areas[_q].observe(_mk_wc(), names="value")

# ── combined side-by-side rows ────────────────────────────────────────────────
call_rows = []
for _c in CALLS:
    _q = _c["quarter"]
    left_panel = widgets.VBox(
        [widgets.HTML(value=make_qa_html(_c)), make_sent_chart(_c)],
        layout=widgets.Layout(width="58%", min_width="0"))
    right_panel = widgets.VBox(
        [widgets.HTML(value=(
             f'<div style="font-family:Georgia;color:#818cf8;font-weight:bold;' +
             f'font-size:1.05em;margin-bottom:12px;">{ANON_MAP.get(_q,_q)}</div>')),
         chip_sliders[_q],
         pred_dropdowns[_q],
         widgets.HTML(value=(
             '<div style="color:#94a3b8;font-size:0.85em;font-family:Georgia;' +
             'margin:10px 0 4px 0;">Rationale <span style="color:#475569;">' +
             '(max 30 words)</span></div>')),
         reason_areas[_q],
         word_counters[_q]],
        layout=widgets.Layout(
            width="40%", min_width="0", padding="16px",
            border="1px solid #334155", border_radius="8px"))
    call_rows.append(widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(margin="0 0 20px 0", align_items="flex-start",
                              grid_gap="16px")))

display(widgets.VBox(call_rows))


In [ ]:
display(HTML('<div class="ag-section-hdr">Round 1 &#8212; Submit your predictions</div>'))


r1_submit_btn = widgets.Button(description="Submit Round 1",
    button_style="success", layout=widgets.Layout(width="200px", height="38px"))
r1_status     = widgets.HTML()
r1_status_box = widgets.Output()

def on_r1_submit(b):
    if not STATE.get("team_name"):
        r1_status.value = '<span style="color:#ef4444;">Register your team first.</span>'; return
    if STATE.get("r1_submitted"):
        r1_status.value = '<span style="color:#f59e0b;">Already submitted.</span>'; return
    for q in QUARTERS:
        if pred_dropdowns[q].value == "-- select --":
            r1_status.value = f'<span style="color:#ef4444;">Select prediction for {ANON_MAP.get(q,q)}.</span>'; return
        wc = len(reason_areas[q].value.split())
        if wc == 0:
            r1_status.value = f'<span style="color:#ef4444;">Enter rationale for {ANON_MAP.get(q,q)}.</span>'; return
        if wc > 30:
            r1_status.value = f'<span style="color:#ef4444;">{ANON_MAP.get(q,q)}: rationale over 30 words ({wc}).</span>'; return
    STATE["chips"] = {q: chip_sliders[q].value for q in QUARTERS}
    payload = {"action": "submit_round1", "team_name": STATE["team_name"],
               "calls": [{"quarter": q, "chips": chip_sliders[q].value,
                          "prediction": pred_dropdowns[q].value,
                          "reasoning": reason_areas[q].value.strip()}
                         for q in QUARTERS]}
    try:
        resp = post_to_script(payload)
        if not resp.get("success"):
            r1_status.value = f'<span style="color:#ef4444;">Error: {resp.get("message")}</span>'; return
    except Exception:
        pass
    STATE["r1_submitted"] = True
    for q in QUARTERS:
        chip_sliders[q].disabled = True; pred_dropdowns[q].disabled = True
        reason_areas[q].disabled = True
    r1_submit_btn.disabled = True
    r1_status.value = '<span style="color:#22c55e;">&#10003; Round 1 submitted.</span>'
    with r1_status_box:
        try:
            st  = get_status(1)
            sub = st.get("submitted_teams", [])
            display(HTML(
                f'<div class="ag-status-box">Teams submitted: ' +
                f'<b style="color:#22c55e;">{len(sub)}</b>/{st.get("total","?")}' +
                f'<br/>{"  ".join(sub)}</div>'))
        except Exception:
            pass

r1_submit_btn.on_click(on_r1_submit)
display(widgets.HBox([r1_submit_btn, r1_status]))
display(r1_status_box)


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:18px 24px;border-radius:8px;border-left:4px solid #f59e0b;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<h2 style='color:#f59e0b;margin:0 0 4px 0;'>Round 2 &#8212; The Images</h2>
<p style='color:#94a3b8;margin:0;font-size:0.95em;'>Each tab below shows the retained infographics from Tesla&#8217;s annual report for the year preceding each earnings call, alongside a signal divergence panel. Classify each call&#8217;s signal and optionally shift one chip before submitting.</p>
</div>


In [ ]:
def make_r2_tab(c):
    q, fy = c['quarter'], c['10k_fy']
    imgs_df = IMG_MFST[q]
    gallery = []
    for _, row in imgs_df.iterrows():
        img_path = row.get('filepath', row.get('path', ''))
        imputed_note = ('<br/><span style="color:#f59e0b;font-size:0.72em;">'
            '↳ supplemented from adjacent year</span>' if row.get('imputed') else '')
        card_html = (
            '<div style="display:inline-block;background:#0f172a;border:1px solid #334155;'
            'border-radius:6px;padding:8px;margin:6px;vertical-align:top;max-width:280px;">'
            f'<div style="color:#818cf8;font-size:0.8em;margin-bottom:4px;">'
            f'{row.get("image_type","Unknown")}</div>'
            f'<div style="color:#94a3b8;font-size:0.75em;">{row.get("data_content","")}</div>'
            f'<div style="color:#475569;font-size:0.72em;">{row.get("section","")}</div>'
            f'{imputed_note}</div>'
        )
        item = [widgets.HTML(value=card_html)]
        try:
            if img_path and Path(img_path).exists():
                item.insert(0, widgets.Image(value=open(img_path,'rb').read(),
                    format='png', layout=widgets.Layout(max_width='260px', max_height='180px')))
        except Exception: pass
        gallery.append(widgets.VBox(item, layout=widgets.Layout(margin='4px')))
    gallery_box = widgets.HBox(gallery, layout=widgets.Layout(flex_flow='row wrap'))
    div_html = (
        f'<div style="font-family:Georgia,serif;">'
        f'<div style="color:#818cf8;font-size:0.9em;margin-bottom:8px;">'
        f'What Tesla reported vs what Tesla showed</div>'
        f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;">'
        f'<div>{build_gaap_table(fy)}</div>'
        f'<div>{build_kpi_table(fy)}</div>'
        f'</div></div>'
    )
    cls_w = widgets.RadioButtons(
        options=[
            'A — Consistent: text, images, and numbers point the same way',
            'B — Divergent: at least one signal contradicts the others',
            'C — Unclear: insufficient information to classify'],
        description='Signal:', style={'description_width':'60px'},
        layout=widgets.Layout(width='98%'))
    return widgets.VBox([
        widgets.HTML(value='<div class="ag-label" style="margin:6px 0 4px 0;">Infographics</div>'),
        gallery_box,
        widgets.HTML(value='<hr style="border-color:#334155;margin:10px 0;">'),
        widgets.HTML(value=div_html),
        widgets.HTML(value='<div class="ag-label" style="margin:12px 0 4px 0;">Signal classification</div>'),
        cls_w,
    ], layout=widgets.Layout(padding='10px')), cls_w

r2_tab      = widgets.Tab()
r2_classify = {}
_tab_contents = []
for _c in CALLS:
    _content, _cls = make_r2_tab(_c)
    _tab_contents.append(_content)
    r2_classify[_c['quarter']] = _cls
r2_tab.children = _tab_contents
for _i, _c in enumerate(CALLS):
    r2_tab.set_title(_i, ANON_MAP.get(_c['quarter'], _c['quarter']))
display(r2_tab)


In [ ]:
display(HTML('<div class="ag-section-hdr">Round 2 &#8212; Optional chip shift &amp; submission</div>'))
shift_from   = widgets.Dropdown(options=ANON_QUARTERS, description='Move 1 chip from:',
    style={'description_width':'140px'})
shift_to     = widgets.Dropdown(options=ANON_QUARTERS, description='to:',
    style={'description_width':'30px'})
shift_btn    = widgets.Button(description='Confirm shift', button_style='warning',
    layout=widgets.Layout(width='140px'))
shift_status = widgets.HTML()

def on_shift_r2(b):
    if STATE.get('r2_chip_shift_used'):
        shift_status.value = '<span style="color:#f59e0b;">Shift already used.</span>'; return
    src = next(q for q,a in ANON_MAP.items() if a==shift_from.value)
    dst = next(q for q,a in ANON_MAP.items() if a==shift_to.value)
    if src == dst:
        shift_status.value = '<span style="color:#ef4444;">Source and destination must differ.</span>'; return
    if STATE['chips'][src] <= 1:
        shift_status.value = f'<span style="color:#ef4444;">{shift_from.value} at minimum (1 chip).</span>'; return
    if STATE['chips'][dst] >= 3:
        shift_status.value = f'<span style="color:#ef4444;">{shift_to.value} at maximum (3 chips).</span>'; return
    STATE['chips'][src] -= 1; STATE['chips'][dst] += 1
    STATE['r2_chip_shift_used'] = True
    shift_btn.disabled=True; shift_from.disabled=True; shift_to.disabled=True
    shift_status.value = (f'<span style="color:#22c55e;">✓ Moved 1 chip: '
        f'{shift_from.value}→{shift_to.value}. New allocations: '
        f'{", ".join(f"{ANON_MAP[q]}:{v}" for q,v in STATE["chips"].items())}</span>')

shift_btn.on_click(on_shift_r2)
display(widgets.HTML('<p style="font-family:Georgia;color:#94a3b8;font-size:0.88em;">'
    'You may move 1 chip once this round. Optional and irreversible.</p>'))
display(widgets.HBox([shift_from, shift_to, shift_btn])); display(shift_status)

r2_submit_btn = widgets.Button(description='Submit Round 2', button_style='success',
    layout=widgets.Layout(width='200px', height='38px'))
r2_status     = widgets.HTML()
r2_status_box = widgets.Output()

def on_r2_submit(b):
    if not STATE.get('r1_submitted'):
        r2_status.value = '<span style="color:#ef4444;">Submit Round 1 first.</span>'; return
    if STATE.get('r2_submitted'):
        r2_status.value = '<span style="color:#f59e0b;">Already submitted.</span>'; return
    payload = {'action':'submit_round2','team_name':STATE['team_name'],
               'calls':[{'quarter':q,'classification':r2_classify[q].value}
                         for q in QUARTERS]}
    try:
        resp = post_to_script(payload)
        if not resp.get('success'):
            r2_status.value = f'<span style="color:#ef4444;">{resp.get("message")}</span>'; return
    except Exception: pass
    STATE['r2_submitted'] = True
    for q in QUARTERS: r2_classify[q].disabled = True
    r2_submit_btn.disabled=True; shift_btn.disabled=True
    r2_status.value = '<span style="color:#22c55e;">✓ Round 2 submitted.</span>'
    with r2_status_box:
        try:
            st = get_status(2); sub = st.get('submitted_teams',[])
            display(HTML(f'<div class="ag-status-box">Round 2 submissions: '
                f'<b style="color:#22c55e;">{len(sub)}</b>/{st.get("total","?")}'
                f'<br/>{"  ".join(sub)}</div>'))
        except Exception: pass

r2_submit_btn.on_click(on_r2_submit)
display(widgets.HBox([r2_submit_btn, r2_status])); display(r2_status_box)


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:18px 24px;border-radius:8px;border-left:4px solid #3b82f6;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<h2 style='color:#3b82f6;margin:0 0 4px 0;'>Round 3 &#8212; The Audio</h2>
<p style='color:#94a3b8;margin:0;font-size:0.95em;'>Paralinguistic features extracted from Elon Musk&#8217;s voice during each call. The silence map shows how much of each 5-minute window is unvoiced; the radar chart profiles nine vocal dimensions against the cross-call average.</p>
</div>


In [ ]:
r3_reason_areas = {}
r3_word_cntrs   = {}

for _c in CALLS:
    _q = _c['quarter']
    display(HTML(f'<div class="ag-section-hdr" style="border-left-color:#3b82f6;">'
                 f'{ANON_MAP.get(_q, _q)}</div>'))
    make_silence_map(_c).show()
    make_radar(_c).show()
    r3_reason_areas[_q] = widgets.Textarea(
        placeholder=f'Update or confirm your Round 1 rationale for {ANON_MAP.get(_q,_q)} (max 30 words) …',
        layout=widgets.Layout(width='720px', height='64px'))
    r3_word_cntrs[_q] = widgets.HTML(value='<div class="ag-word-counter">0 / 30 words</div>')
    def _mk_wc3(_q_=_q):
        def _upd(ch):
            wc = len(ch['new'].split())
            col = '#ef4444' if wc > 30 else '#94a3b8'
            r3_word_cntrs[_q_].value = (f'<div class="ag-word-counter" '
                f'style="color:{col};">{wc} / 30 words</div>')
        return _upd
    r3_reason_areas[_q].observe(_mk_wc3(), names='value')
    display(widgets.HTML(f'<span style="color:#94a3b8;font-size:0.85em;">'
                          f'Update reasoning for {ANON_MAP.get(_q,_q)}:</span>'))
    display(widgets.VBox([r3_reason_areas[_q], r3_word_cntrs[_q]]))
    display(HTML('<hr style="border-color:#1e293b;margin:12px 0;">'))


In [ ]:
display(HTML('<div class="ag-section-hdr" style="border-left-color:#3b82f6;">'
             'Round 3 &#8212; Optional chip shift &amp; final submission</div>'))
r3_shift_from   = widgets.Dropdown(options=ANON_QUARTERS, description='Move 1 chip from:',
    style={'description_width':'140px'})
r3_shift_to     = widgets.Dropdown(options=ANON_QUARTERS, description='to:',
    style={'description_width':'30px'})
r3_shift_btn    = widgets.Button(description='Confirm shift', button_style='warning',
    layout=widgets.Layout(width='140px'))
r3_shift_status = widgets.HTML()

def on_shift_r3(b):
    if STATE.get('r3_chip_shift_used'):
        r3_shift_status.value='<span style="color:#f59e0b;">Shift already used.</span>'; return
    src = next(q for q,a in ANON_MAP.items() if a==r3_shift_from.value)
    dst = next(q for q,a in ANON_MAP.items() if a==r3_shift_to.value)
    if src==dst:
        r3_shift_status.value='<span style="color:#ef4444;">Must differ.</span>'; return
    if STATE['chips'][src] <= 1:
        r3_shift_status.value=f'<span style="color:#ef4444;">{r3_shift_from.value} at min (1 chip).</span>'; return
    if STATE['chips'][dst] >= 3:
        r3_shift_status.value=f'<span style="color:#ef4444;">{r3_shift_to.value} at max (3 chips).</span>'; return
    STATE['chips'][src]-=1; STATE['chips'][dst]+=1
    STATE['r3_chip_shift_used']=True
    r3_shift_btn.disabled=True; r3_shift_from.disabled=True; r3_shift_to.disabled=True
    r3_shift_status.value=(f'<span style="color:#22c55e;">✓ Moved: '
        f'{r3_shift_from.value}→{r3_shift_to.value}. Final chips: '
        f'{", ".join(f"{ANON_MAP[q]}:{v}" for q,v in STATE["chips"].items())}</span>')

r3_shift_btn.on_click(on_shift_r3)
display(widgets.HBox([r3_shift_from, r3_shift_to, r3_shift_btn]))
display(r3_shift_status)

r3_submit_btn = widgets.Button(description='Submit Final Predictions',
    button_style='success', layout=widgets.Layout(width='240px', height='38px'))
r3_status     = widgets.HTML()
r3_status_box = widgets.Output()

def on_r3_submit(b):
    if not STATE.get('r2_submitted'):
        r3_status.value='<span style="color:#ef4444;">Submit Round 2 first.</span>'; return
    if STATE.get('r3_submitted'):
        r3_status.value='<span style="color:#f59e0b;">Already submitted.</span>'; return
    for q in QUARTERS:
        if len(r3_reason_areas[q].value.split()) > 30:
            r3_status.value=f'<span style="color:#ef4444;">{ANON_MAP.get(q,q)} over 30 words.</span>'; return
    payload = {'action':'submit_round3','team_name':STATE['team_name'],
               'calls':[{'quarter':q,'chips':STATE['chips'][q],
                          'prediction':pred_dropdowns[q].value,
                          'reasoning':(r3_reason_areas[q].value.strip()
                                       or reason_areas[q].value.strip())}
                         for q in QUARTERS]}
    try:
        resp = post_to_script(payload)
        if not resp.get('success'):
            r3_status.value=f'<span style="color:#ef4444;">{resp.get("message")}</span>'; return
    except Exception: pass
    STATE['r3_submitted']=True
    for q in QUARTERS: r3_reason_areas[q].disabled=True
    r3_submit_btn.disabled=True; r3_shift_btn.disabled=True
    r3_status.value='<span style="color:#22c55e;">✓ Final predictions submitted. Scroll down for The Reveal.</span>'
    with r3_status_box:
        try:
            st=get_status(3); sub=st.get('submitted_teams',[])
            display(HTML(f'<div class="ag-status-box">Final submissions: '
                f'<b style="color:#22c55e;">{len(sub)}</b>/{st.get("total","?")}'
                f'<br/>{"  ".join(sub)}</div>'))
        except Exception: pass

r3_submit_btn.on_click(on_r3_submit)
display(widgets.HBox([r3_submit_btn, r3_status])); display(r3_status_box)


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:18px 24px;border-radius:8px;border-left:4px solid #22c55e;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<h2 style='color:#22c55e;margin:0 0 4px 0;'>The Reveal</h2>
<p style='color:#94a3b8;margin:0;font-size:0.95em;'>Run this cell when the facilitator signals. The actual 5-day abnormal returns are displayed, predictions scored, and the debrief matrix shown.</p>
</div>


In [ ]:
_qtrs   = [c['quarter'] for c in CALLS]
_anons  = [ANON_MAP.get(q, q) for q in _qtrs]
_rets   = [c['returns']['abnormal_ret_pct'] for c in CALLS]
_dirs   = [c['returns']['direction'] for c in CALLS]
_colors = [CLR['ret_positive'] if d=='positive' else CLR['ret_negative'] for d in _dirs]
_labels = [f"{a}<br>({q})<br>{r:+.1f}%" for a, q, r in zip(_anons, _qtrs, _rets)]

_frames = [go.Frame(
    data=[go.Bar(x=_anons[:i+1], y=_rets[:i+1], text=_labels[:i+1],
               textposition='outside', textfont=dict(color='#e2e8f0', size=11),
               marker_color=_colors[:i+1])],
    name=str(i)) for i in range(len(_qtrs))]

_fig_reveal = go.Figure(
    data=[go.Bar(x=[], y=[], marker_color=[])],
    frames=_frames,
    layout=go.Layout(
        title=dict(text='5-Day Abnormal Returns — Actual Outcomes',
                   font=dict(color='#e2e8f0', size=15)),
        xaxis=dict(color='#94a3b8', gridcolor='#1e293b'),
        yaxis=dict(title='Abnormal return (%)', color='#94a3b8',
                   gridcolor='#1e293b', zeroline=True, zerolinecolor='#475569'),
        plot_bgcolor='#0f172a', paper_bgcolor='#1e293b',
        font=dict(color='#e2e8f0'), height=400,
        updatemenus=[{'type':'buttons','showactive':False,
            'buttons':[{'label':'▶  Reveal','method':'animate',
                'args':[None,{'frame':{'duration':1200,'redraw':True},
                              'transition':{'duration':600},'fromcurrent':True}]}],
            'x':0.5,'xanchor':'center','y':-0.12,'yanchor':'top'}]))
_fig_reveal.show()


# ── Reveal price chart ───────────────────────────────────────────────────────
display(HTML('<div class="ag-section-hdr">Where your calls sit on the Tesla timeline</div>'))
try:
    display(Image(filename=f'{DRIVE_ROOT}/TSLA_price_timeline_reveal.png', width=900))
except Exception:
    display(HTML('<p style="color:#94a3b8;">Reveal chart not found.</p>'))
# ── Scoring ──────────────────────────────────────────────────────────────────
import time; time.sleep(0.3)
display(HTML('<div class="ag-section-hdr">Scoring</div>'))

_score = 0
_breakdown = []
for _c in CALLS:
    _q = _c['quarter']
    _actual = _c['returns']['direction']
    _pred   = pred_dropdowns[_q].value
    _chips  = STATE['chips'][_q]
    _ok = ((_pred=='Stock rose' and _actual=='positive') or
           (_pred=='Stock fell' and _actual=='negative'))
    _pts = _chips if _ok else -_chips
    _score += _pts
    _breakdown.append(dict(call=ANON_MAP.get(_q,_q), actual_q=_q, prediction=_pred, actual=_actual,
                           chips=_chips, pts=_pts, ok=_ok))

_rows_html = ''
for _b in _breakdown:
    _col = '#22c55e' if _b['ok'] else '#ef4444'
    _act_str = 'positive ↑' if _b['actual']=='positive' else 'negative ↓'
    _rows_html += (
        f'<tr><td>{_b["call"]}</td><td>{_b["prediction"]}</td>'
        f'<td>{_act_str}</td><td>{_b["chips"]}</td>'
        f'<td style="color:{_col};">{"+" if _b["ok"] else ""}{_b["pts"]}</td></tr>')
_tot_col = '#22c55e' if _score >= 0 else '#ef4444'
display(HTML(
    f'<table class="ag-table">'
    f'<tr><th>Call</th><th>Your prediction</th><th>Actual</th>'
    f'<th>Chips</th><th>Points</th></tr>'
    f'{_rows_html}'
    f'<tr style="border-top:2px solid #334155;">'
    f'<td colspan="4" style="text-align:right;color:#94a3b8;">Total</td>'
    f'<td style="color:{_tot_col};font-weight:bold;">{"+" if _score>=0 else ""}{_score}</td></tr>'
    f'</table>'))

# ── Upset calls ───────────────────────────────────────────────────────────────
_upset = [c['quarter'] for c in CALLS
          if (c['excerpt']['musk_net_sent']>0.01 and c['returns']['direction']=='negative')
          or (c['excerpt']['musk_net_sent']<-0.01 and c['returns']['direction']=='positive')]
if _upset:
    display(HTML(
        '<div style="background:#1e293b;border:1px solid #f59e0b;border-radius:8px;'
        'padding:14px 20px;margin:12px 0;font-family:Georgia;">'
        '<p style="color:#f59e0b;margin:0 0 8px 0;font-weight:bold;">'
        f'Upset calls: {", ".join(_upset)}</p>'
        '<p style="color:#e2e8f0;font-size:0.9em;">Calls where Musk\'s LM sentiment '
        'pointed in the opposite direction to the actual abnormal return. '
        'These are where acoustic and visual signals carry the most information value.</p>'
        '</div>'))

# ── Debrief matrix ────────────────────────────────────────────────────────────
display(HTML('<div class="ag-section-hdr">Debrief matrix — signal coherence</div>'))

_matrix_rows = ''
for _c in CALLS:
    _q      = _c['quarter']
    _net    = _c['excerpt']['musk_net_sent']
    _lm_dir = 'positive' if _net>0.005 else ('negative' if _net<-0.005 else 'neutral')
    _row    = SUMMARY[SUMMARY['call']==f'TSLA_{_q}']
    _fou    = _row['fraction_of_unvoiced_mean_musk'].values
    _aco    = ('negative' if (len(_fou) and not np.isnan(_fou[0]) and _fou[0]>FOU_MEAN)
               else 'positive' if len(_fou) else 'n/a')
    _actual = _c['returns']['direction']
    def _bg(sig, actual=_actual):
        if sig in ('neutral','n/a'): return '#1e293b'
        return '#14532d' if sig==actual else '#450a0a'
    def _arr(sig):
        if sig=='n/a':       return '<span class="ag-na">n/a</span>'
        if sig=='positive':  return '↑ positive'
        if sig=='neutral':   return '→ neutral'
        return '↓ negative'
    _act_col = '#22c55e' if _actual=='positive' else '#ef4444'
    _matrix_rows += (
        f'<tr><td style="color:#818cf8;font-weight:bold;">{_q}</td>'
        f'<td style="background:{_bg(_lm_dir)};">{_arr(_lm_dir)}</td>'
        f'<td style="background:{_bg(_aco)};">{_arr(_aco)}</td>'
        f'<td style="background:#1e293b;"><span class="ag-na">n/a</span></td>'
        f'<td style="color:{_act_col};font-weight:bold;">{_arr(_actual)}</td></tr>')

display(HTML(
    '<p style="color:#94a3b8;font-size:0.85em;margin:4px 0 8px 0;">'
    'Green cell = signal matched outcome. Red = contradicted. '
    'Images column shows n/a (EDGAR HTML cover photos only).</p>'
    '<table class="ag-table">'
    '<tr><th>Call</th><th>Text (LM)</th><th>Audio (FoU)</th>'
    '<th>Images</th><th>Actual outcome</th></tr>'
    f'{_matrix_rows}'
    '</table>'))


<div style='background:linear-gradient(90deg,#1e1b4b,#0f172a);padding:18px 24px;border-radius:8px;border-left:4px solid #94a3b8;margin:20px 0 12px 0;font-family:Georgia,serif;'>
<h2 style='color:#94a3b8;margin:0 0 4px 0;'>Individual Reflection</h2>
<p style='color:#94a3b8;margin:0;font-size:0.95em;'>This section is individual. Your responses are not shared with other teams and will not appear on the leaderboard. Submitted to a personal tab of the session spreadsheet, keyed by your team name and a timestamp.</p>
</div>


In [ ]:
_questions = [
    'Which modality (text, images, or audio) was most useful for predicting outcomes, and why?',
    'Describe a call where the three modalities sent conflicting signals. How did you resolve the conflict?',
    'If you were building an automated signal-extraction system for earnings calls, '
    'which single feature would you prioritise and why?',
]
_ref_areas = [widgets.Textarea(placeholder=q,
    layout=widgets.Layout(width='720px', height='90px')) for q in _questions]

ref_submit_btn = widgets.Button(description='Submit reflection',
    layout=widgets.Layout(width='200px'))
ref_status = widgets.HTML()

def on_ref_submit(b):
    if not STATE.get('team_name'):
        ref_status.value='<span style="color:#ef4444;">Register first.</span>'; return
    payload = {'action':'submit_reflection','team_name':STATE['team_name'],
               'q1':_ref_areas[0].value.strip(),
               'q2':_ref_areas[1].value.strip(),
               'q3':_ref_areas[2].value.strip()}
    try:
        resp = post_to_script(payload)
        if resp.get('success'):
            ref_submit_btn.disabled=True
            ref_status.value='<span style="color:#22c55e;">✓ Reflection submitted.</span>'
        else:
            ref_status.value=f'<span style="color:#ef4444;">{resp}</span>'
    except Exception:
        ref_submit_btn.disabled=True
        ref_status.value='<span style="color:#f59e0b;">⚠ Offline — thank you.</span>'

ref_submit_btn.on_click(on_ref_submit)

for _i, (_qlbl, _w) in enumerate(zip(_questions, _ref_areas), 1):
    display(HTML(f'<div style="font-family:Georgia;color:#e2e8f0;margin:12px 0 4px 0;">'
                 f'{_i}. {_qlbl}</div>'))
    display(_w)
display(widgets.HBox([ref_submit_btn, ref_status]))
